# Missionários em segurança com canibais
O intuito desse trabalho é gerenciar o uso do barco para que todos os missionários da paróquia do Frei Sardinha consiga atravessar o rio em segurança na companhia dos queridos indígenas Caetés. Mas para garantir a segurança de todos, principalmente dos missionários, existem algumas regras que devem ser respeitadas.

Primeiramente, o problema considera duas margens do rio: a margem inicial (esquerda) e a margem de destino (direita). Todos os indivíduos começam na margem inicial e devem ser transportados para a margem de destino.

O transporte é realizado por meio de um barco, que possui capacidade limitada, podendo transportar no máximo um número fixo de pessoas por travessia.

## Regras

As regras que regem o problema são as seguintes:

- O barco deve sempre transportar pelo menos uma pessoa, o barco nunca se desloca sozinho.
- Em nenhuma das margens do rio pode haver um número de indígenas maior que o de missionários, caso exista pelo menos um missionário presente naquela margem. Caso contrário, os missionários seriam colocados em risco.

# Índice

1. [Como usar](#Como-usar)
2. [Definições iniciais](#Definições-iniciais)
3. [Validação](#Validação)
4. [BFS - Busca em Largura](#BFS---Busca-em-Largura)
5. [DFS - Busca em Profundidade](#DFS---Busca-em-Profundidade)
6. [DLS - Busca em Profundidade Limitada](#DLS---Busca-em-Profundidade-Limitada)
7. [Playground](#Playground)

## Como usar
com todas as células compiladas é possível utilizar no final do arquivo o playground, todos os 3 cenários estão prontos com métricas para serem retornados, é possível configurar o número de missionários, de canibais e o tamanho do barco em todos os algorítmos, no de Busca por Profundidade limitada também é possível definir, o limite inicial e o limite máximo.

## Definições iniciais

In [53]:
from time import perf_counter
from collections import deque

# possibilita que os parâmetros possam ser redefinidos conforme necessidade do cenário
def set_initial_state(missionaries=3, cannibals=3, boat="left"):
    state = {
        "left": (missionaries, cannibals),
        "right": (0, 0),
        "boat": boat
    }
    return state
initial_state = set_initial_state()

#melhora a qualidade de exibição das métricas e estatísticas
def print_metrics(metrics):
    match(metrics):
        case {"tipo": "dls", "sucesso": False}:
            print(f"❌ Tempo de execução: {metrics['tempo_de_execução']:.6f} s")
        case _:
            print("\n📊 MÉTRICAS DE EXECUÇÃO")
            print("=" * 35)
        
            print(f"Sucesso: {'✅' if metrics['sucesso'] else '❌'}")
        
            print(f"Tempo de execução: {metrics['tempo_de_execução']:.6f} s")
        
            print(f"Estados verificados: {metrics['estados_verificados']:,}".replace(",", "."))
            print(f"Nós gerados: {metrics['nodes_gerados']:,}".replace(",", "."))
            print(f"Nós expandidos: {metrics['nodes_expandidos']:,}".replace(",", "."))
        
            print(f"Tamanho máximo da {metrics['estrutura']}: {metrics['tamanho_maximo_estrutura']:,}".replace(",", "."))
        
            profundidade = metrics["profundidade_solucao"]
            if profundidade is not None:
                print(f"Profundidade da solução: {profundidade}")
            else:
                print("Profundidade da solução: —")
        
            print("=" * 35)

### Validação
A validação consiste em verificar em cada margem, se há mais canibais que missionarios.

In [2]:
def is_valid(state):
    left_missionaries, left_cannibals = state["left"]
    right_missionaries, right_cannibals = state["right"]

    if min(left_missionaries, left_cannibals, right_missionaries, right_cannibals) < 0:
        return False

    if left_missionaries > 0 and left_cannibals > left_missionaries:
        return False

    if right_missionaries > 0 and right_cannibals > right_missionaries:
        return False

    return True

### Função sucessor
É listado todos os movimentos possíveis de acordo com a quantidade de lugares no barco, por meio da soma do valor dos index, é possível comparar quantas pessoas de cada cultura cabe no espaço do barco.

In [3]:
def possible_moves(boat_capacity):
    moves = []
    
    for m in range(boat_capacity + 1):
        for c in range(boat_capacity + 1):
            total = m + c
            
            if 1 <= total <= boat_capacity:
                moves.append((m, c))
    
    return moves

In [4]:
possible_moves(2) # Um exemplo de uso com o máximo de duas pessoas no barco

[(0, 1), (0, 2), (1, 0), (1, 1), (2, 0)]

Com isso, podemos finalmente criar a função que aplica os movimentos:

In [5]:
def apply_move(state, move):
    missionaries, cannibals = move
    left_missionaries, left_cannibals = state["left"]
    right_missionaries, right_cannibals = state["right"]

    if state["boat"] == "left":
        return {
            "left": (left_missionaries - missionaries, left_cannibals - cannibals),
            "right": (right_missionaries + missionaries, right_cannibals + cannibals),
            "boat": "right"
        }
    else:
        return {
            "left": (left_missionaries + missionaries, left_cannibals + cannibals),
            "right": (right_missionaries - missionaries, right_cannibals - cannibals),
            "boat": "left"
        }

In [6]:
# Test
initial_state = set_initial_state(3, 3)
new_state = apply_move(initial_state, (2, 1))
print(new_state)
print("Válido?", is_valid(new_state))

{'left': (1, 2), 'right': (2, 1), 'boat': 'right'}
Válido? False


### Explicação da função sucessor
A função sucessor é responsável por gerar, a partir de um estado atual, todos os próximos estados possíveis que podem ser alcançados em **uma única travessia** do barco. Para isso, ela combina três etapas já definidas anteriormente:

1. Obtém todos os movimentos possíveis por meio da função `possible_moves()` (combinações de missionários e canibais que cabem no barco).
2. Aplica cada movimento ao estado atual com a função `apply_move(state, move)`, produzindo um novo estado candidato.
3. Valida esse novo estado com `is_valid(new_state)`, descartando automaticamente estados que violam as regras do problema.

Ao final, a função retorna uma lista de pares `(movimento, novo_estado)`. Essa estrutura é essencial para os algoritmos de busca (como o BFS), pois permite explorar o espaço de estados de forma sistemática, mantendo apenas transições válidas e seguras.

In [7]:
def successors(state, boat_capacity):
    result = []

    for move in possible_moves(boat_capacity):
        new_state = apply_move(state, move)

        if is_valid(new_state):
            result.append((move, new_state))

    return result

In [8]:
def is_goal(state):
    return state["left"] == (0, 0)

## BFS - Busca em Largura
Com o auxílo de uma fila, o algorítmo percorre linha por linha da arvore, mantendo o mesmo nível na busca por uma solução, tende a percorrer a árvore toda e só para quando encontra um resultado, o resultado tende a ser o mais curto em passos.

In [32]:
# Recebe um estado inicial e a capacidade do barco, que podem
def solve_bfs(initial_state, boat_capacity):
    start_time = perf_counter()
    queue = deque([{
        "state": initial_state,
        "path": [],
        "movements": [(0,0)]
    }])

    visited = {
        (
            initial_state["left"],
            initial_state["right"],
            initial_state["boat"]
        )
    }

    generated_nodes = 1
    expanded_nodes = 0
    max_queue_size = 1

    while queue:
        max_queue_size = max(max_queue_size, len(queue))
        
        node = queue.popleft()
        expanded_nodes += 1

        state = node["state"]
        path = node["path"]
        movements = node["movements"]

        if is_goal(state):
            end_time = perf_counter()
            solution = path + [state]

            success_metrics = {
                "sucesso": True,
                "tempo_de_execução": end_time - start_time,
                "estados_verificados": len(visited),
                "nodes_gerados": generated_nodes,
                "nodes_expandidos": expanded_nodes,
                "tamanho_maximo_estrutura": max_queue_size,
                "estrutura": "fila",
                "profundidade_solucao": len(solution) - 1,
                "tipo": "bfs"
            }
            
            return solution, movements, success_metrics

        #obtem uma lista de movimentos e estados que esses movimentos geram
        for move, new_state in successors(state, boat_capacity):
            state_key = (
                new_state["left"],
                new_state["right"],
                new_state["boat"]
            )

            # verifica se o estado analisado é um estado que nunca foi visitado, 
            # se não for, esse estado é adicionado na fila pra processamento futuro
            if state_key not in visited:
                visited.add(state_key)
                generated_nodes += 1

                queue.append({
                    "state": new_state,
                    "path": path + [state],
                    "movements": movements + [move]
                })
                
    end_time = perf_counter()

    failed_metrics = {
        "sucesso": False,
        "tempo_de_execução": end_time - start_time,
        "estados_verificados": len(visited),
        "nodes_gerados": generated_nodes,
        "nodes_expandidos": expanded_nodes,
        "tamanho_maximo_estrutura": max_queue_size,
        "estrutura": "fila",
        "profundidade_solucao": None,
        "tipo": "bfs"
    }


    return None, None, failed_metrics

In [58]:
def scenario_bfs(missionaries = 3, cannibals = 3, boat_size = 2):
    initial_state = set_initial_state(missionaries, cannibals)
    solution, movements, metrics = solve_bfs(initial_state, boat_size)
    
    if solution and movements:
        
        total_steps = len(movements) - 1  # ignorando (0,0)
        print(f"Resolvido em {total_steps} passos.")
    
        if total_steps > 20:
            print("Solução muito longa, não será exibida.")
        else:
            print("\nGabarito da travessia:")
            boat_side = "left"
        
            for step, move in enumerate(movements[1:], start=1):
                missionaries, cannibals = move
                origin = "esquerda" if boat_side == "left" else "direita"
                destination = "direita" if boat_side == "left" else "esquerda"
        
                missionaries_label = "missionário" if missionaries == 1 else "missionários"
                cannibals_label = "canibal" if cannibals == 1 else "canibais"
        
                print(
                    f"Passo {step}: levar {missionaries} {missionaries_label} e "
                    f"{cannibals} {cannibals_label} da margem {origin} para a margem {destination}."
                )
        
                boat_side = "right" if boat_side == "left" else "left"
    else:
        print("Nenhuma solução encontrada")
    
    print_metrics(metrics)
scenario_bfs()

Resolvido em 11 passos.

Gabarito da travessia:
Passo 1: levar 0 missionários e 2 canibais da margem esquerda para a margem direita.
Passo 2: levar 0 missionários e 1 canibal da margem direita para a margem esquerda.
Passo 3: levar 0 missionários e 2 canibais da margem esquerda para a margem direita.
Passo 4: levar 0 missionários e 1 canibal da margem direita para a margem esquerda.
Passo 5: levar 2 missionários e 0 canibais da margem esquerda para a margem direita.
Passo 6: levar 1 missionário e 1 canibal da margem direita para a margem esquerda.
Passo 7: levar 2 missionários e 0 canibais da margem esquerda para a margem direita.
Passo 8: levar 0 missionários e 1 canibal da margem direita para a margem esquerda.
Passo 9: levar 0 missionários e 2 canibais da margem esquerda para a margem direita.
Passo 10: levar 0 missionários e 1 canibal da margem direita para a margem esquerda.
Passo 11: levar 0 missionários e 2 canibais da margem esquerda para a margem direita.

📊 MÉTRICAS DE EXECUÇ

## DFS - Busca em Profundidade
A busca em profundidade explora um caminho até o fim antes de voltar (backtracking) para tentar alternativas. Aqui, usamos uma pilha para controlar os estados pendentes e também evitamos revisitar estados já explorados.

In [38]:
def solve_dfs(initial_state, boat_capacity):
    start_time = perf_counter()
    stack = [{
        "state": initial_state,
        "path": [],
        "movements": [(0, 0)]
    }]

    visited = {
        (
            initial_state["left"],
            initial_state["right"],
            initial_state["boat"]
        )
    }

    generated_nodes = 1
    expanded_nodes = 0
    max_stack_size = 1
    
    while stack:
        max_stack_size = max(max_stack_size, len(stack))
        
        node = stack.pop()
        expanded_nodes += 1

        state = node["state"]
        path = node["path"]
        movements = node["movements"]

        if is_goal(state):
            end_time = perf_counter()
            solution = path + [state]
            
            success_metrics = {
                "sucesso": True,
                "tempo_de_execução": end_time - start_time,
                "estados_verificados": len(visited),
                "nodes_gerados": generated_nodes,
                "nodes_expandidos": expanded_nodes,
                "tamanho_maximo_estrutura": max_stack_size,
                "estrutura": "pilha",
                "profundidade_solucao": len(solution) - 1,
                "tipo": "dfs"
            }
            
            return solution, movements, success_metrics

        for move, new_state in successors(state, boat_capacity):
            state_key = (
                new_state["left"],
                new_state["right"],
                new_state["boat"]
            )

            if state_key not in visited:
                visited.add(state_key)
                stack.append({
                    "state": new_state,
                    "path": path + [state],
                    "movements": movements + [move]
                })
                
                generated_nodes += 1


    
    end_time = perf_counter()
    
    failed_metrics = {
        "sucesso": False,
        "tempo_de_execução": end_time - start_time,
        "estados_verificados": len(visited),
        "nodes_gerados": generated_nodes,
        "nodes_expandidos": expanded_nodes,
        "tamanho_maximo_estrutura": max_stack_size,
        "estrutura": "pilha",
        "profundidade_solucao": None,
        "tipo": "dfs"
    }
    return None, failed_metrics 

In [55]:
def scenario_dfs(missionaries = 3, cannibals = 3, boat_size = 2):
    initial_state = set_initial_state(missionaries, cannibals)
    solution_dfs, movements_dfs, metrics = solve_dfs(initial_state, boat_size)
    
    if solution_dfs and movements_dfs:
        
        total_steps = len(movements_dfs) - 1  # ignorando (0,0)
        print(f"Resolvido em {total_steps} passos.")
    
        if total_steps > 20:
            print("Solução muito longa, não será exibida.")
        else:
            print("\nGabarito da travessia (DFS):")
            boat_side = "left"
        
            for step, move in enumerate(movements_dfs[1:], start=1):
                missionaries, cannibals = move
                origin = "esquerda" if boat_side == "left" else "direita"
                destination = "direita" if boat_side == "left" else "esquerda"
        
                missionaries_label = "missionário" if missionaries == 1 else "missionários"
                cannibals_label = "canibal" if cannibals == 1 else "canibais"
        
                print(
                    f"Passo {step}: levar {missionaries} {missionaries_label} e "
                    f"{cannibals} {cannibals_label} da margem {origin} para a margem {destination}."
                )
        
                boat_side = "right" if boat_side == "left" else "left"
    else:
        print("Nenhuma solução encontrada com DFS")
    
    print_metrics(metrics)
scenario_dfs()

Resolvido em 11 passos.

Gabarito da travessia (DFS):
Passo 1: levar 1 missionário e 1 canibal da margem esquerda para a margem direita.
Passo 2: levar 1 missionário e 0 canibais da margem direita para a margem esquerda.
Passo 3: levar 0 missionários e 2 canibais da margem esquerda para a margem direita.
Passo 4: levar 0 missionários e 1 canibal da margem direita para a margem esquerda.
Passo 5: levar 2 missionários e 0 canibais da margem esquerda para a margem direita.
Passo 6: levar 1 missionário e 1 canibal da margem direita para a margem esquerda.
Passo 7: levar 2 missionários e 0 canibais da margem esquerda para a margem direita.
Passo 8: levar 0 missionários e 1 canibal da margem direita para a margem esquerda.
Passo 9: levar 0 missionários e 2 canibais da margem esquerda para a margem direita.
Passo 10: levar 1 missionário e 0 canibais da margem direita para a margem esquerda.
Passo 11: levar 1 missionário e 1 canibal da margem esquerda para a margem direita.

📊 MÉTRICAS DE EXEC

## DLS - Busca em Profundidade Limitada
A busca em profundidade limitada (Depth-Limited Search) funciona como a DFS, mas com um limite máximo de profundidade. Isso evita que a busca avance indefinidamente em ramos muito longos e permite controlar o custo de exploração.

In [47]:
def solve_dls(initial_state, depth_limit, boat_capacity):
    start_time = perf_counter()
    def state_to_key(state):
        return (state["left"], state["right"], state["boat"])

    initial_key = state_to_key(initial_state)
    stack = [{
        "state": initial_state,
        "path": [],
        "movements": [(0, 0)],
        "depth": 0,
        "path_keys": {initial_key}
    }]

    generated_nodes = 1
    expanded_nodes = 0
    max_stack_size = 1

    while stack:
        max_stack_size = max(max_stack_size,len(stack))
        
        node = stack.pop()
        expanded_nodes += 1

        state = node["state"]
        path = node["path"]
        movements = node["movements"]
        depth = node["depth"]
        path_keys = node["path_keys"]

        if is_goal(state):
            end_time = perf_counter()
            solution = path + [state]

            success_metrics = {
                "sucesso": True,
                "tempo_de_execução": end_time - start_time,
                "estados_verificados": len(path_keys),
                "nodes_gerados": generated_nodes,
                "nodes_expandidos": expanded_nodes,
                "tamanho_maximo_estrutura": max_stack_size,
                "estrutura": "pilha",
                "profundidade_solucao": len(solution) - 1,
                "tipo": "dls"
            }
            
            return solution, movements, success_metrics

        if depth >= depth_limit:
            continue

        for move, new_state in successors(state, boat_capacity):
            state_key = state_to_key(new_state)

            # Evita ciclos no caminho atual da busca em profundidade
            if state_key in path_keys:
                continue

            stack.append({
                "state": new_state,
                "path": path + [state],
                "movements": movements + [move],
                "depth": depth + 1,
                "path_keys": path_keys | {state_key}
            })
            
            generated_nodes += 1

    end_time = perf_counter()
    
    failed_metrics = {
        "sucesso": False,
        "tempo_de_execução": end_time - start_time,
        "estados_verificados": len(path_keys),
        "nodes_gerados": generated_nodes,
        "nodes_expandidos": expanded_nodes,
        "tamanho_maximo_estrutura": max_stack_size,
        "estrutura": "pilha",
        "profundidade_solucao": None,
        "tipo": "dls"
    }
    return None, None, failed_metrics 

In [57]:
def scenario_dls(missionaries = 3, cannibals = 3, boat_size = 2, depth_limit = 0, max_depth_limit = 500):

    initial_state = set_initial_state(missionaries, cannibals)
    solution_dls, movements_dls = None, None
    
    while depth_limit <= max_depth_limit:
        solution_dls, movements_dls, metrics = solve_dls(initial_state, depth_limit, boat_size)
    
        if solution_dls is None:
            print(f"depth_limit = {depth_limit}: nenhuma solução encontrada.")
            print_metrics(metrics)
            depth_limit += 1
            continue
    
        print(f"depth_limit = {depth_limit}: solução encontrada!")
        break
    
    if solution_dls is None:
        print(
            f"\nNenhuma solução encontrada até depth_limit = {max_depth_limit}."
        )
    else:
        print("\nGabarito da travessia (DLS):")
        boat_side = "left"
    
        for step, move in enumerate(movements_dls[1:], start=1):
            missionaries, cannibals = move
            origin = "esquerda" if boat_side == "left" else "direita"
            destination = "direita" if boat_side == "left" else "esquerda"
    
            missionaries_label = "missionário" if missionaries == 1 else "missionários"
            cannibals_label = "canibal" if cannibals == 1 else "canibais"
    
            print(
                f"Passo {step}: levar {missionaries} {missionaries_label} e "
                f"{cannibals} {cannibals_label} da margem {origin} para a margem {destination}."
            )
    
            boat_side = "right" if boat_side == "left" else "left"
        print_metrics(metrics)
scenario_dls()

depth_limit = 0: nenhuma solução encontrada.
❌ Tempo de execução: 0.000009 s
depth_limit = 1: nenhuma solução encontrada.
❌ Tempo de execução: 0.000040 s
depth_limit = 2: nenhuma solução encontrada.
❌ Tempo de execução: 0.000054 s
depth_limit = 3: nenhuma solução encontrada.
❌ Tempo de execução: 0.000078 s
depth_limit = 4: nenhuma solução encontrada.
❌ Tempo de execução: 0.000101 s
depth_limit = 5: nenhuma solução encontrada.
❌ Tempo de execução: 0.000117 s
depth_limit = 6: nenhuma solução encontrada.
❌ Tempo de execução: 0.000135 s
depth_limit = 7: nenhuma solução encontrada.
❌ Tempo de execução: 0.000154 s
depth_limit = 8: nenhuma solução encontrada.
❌ Tempo de execução: 0.000158 s
depth_limit = 9: nenhuma solução encontrada.
❌ Tempo de execução: 0.000178 s
depth_limit = 10: nenhuma solução encontrada.
❌ Tempo de execução: 0.000234 s
depth_limit = 11: solução encontrada!

Gabarito da travessia (DLS):
Passo 1: levar 1 missionário e 1 canibal da margem esquerda para a margem direita.
P

## Playground

O playground deste notebook expõe três funções públicas para testar diferentes estratégias de busca no problema dos missionários e canibais.

### Funções públicas

#### `scenario_bfs(missionaries=3, cannibals=3, boat_size=2)`

Executa o problema usando **Busca em Largura (BFS)**.

Parâmetros:
- `missionaries`: quantidade inicial de missionários na margem esquerda
- `cannibals`: quantidade inicial de canibais na margem esquerda
- `boat_size`: capacidade máxima do barco

---

#### `scenario_dfs(missionaries=3, cannibals=3, boat_size=2)`

Executa o problema usando **Busca em Profundidade (DFS)**.

Parâmetros:
- `missionaries`: quantidade inicial de missionários na margem esquerda
- `cannibals`: quantidade inicial de canibais na margem esquerda
- `boat_size`: capacidade máxima do barco

---

#### `scenario_dls(missionaries=3, cannibals=3, boat_size=2, depth_limit=0, max_depth_limit=500)`

Executa o problema usando **Busca em Profundidade Limitada (DLS)**, tentando encontrar solução a partir de um limite inicial de profundidade.

Parâmetros:
- `missionaries`: quantidade inicial de missionários na margem esquerda
- `cannibals`: quantidade inicial de canibais na margem esquerda
- `boat_size`: capacidade máxima do barco
- `depth_limit`: limite inicial de profundidade
- `max_depth_limit`: limite máximo de profundidade a ser testado

### Exemplos de uso

```python
scenario_bfs()
scenario_dfs()

scenario_bfs(missionaries=4, cannibals=4)
scenario_dfs(4, 3, 3)
scenario_dls(missionaries=3, cannibals=3, boat_size=2, depth_limit=0, max_depth_limit=20)

In [60]:
scenario_bfs(missionaries=41)

Resolvido em 85 passos.
Solução muito longa, não será exibida.

📊 MÉTRICAS DE EXECUÇÃO
Sucesso: ✅
Tempo de execução: 0.003496 s
Estados verificados: 318
Nós gerados: 318
Nós expandidos: 318
Tamanho máximo da fila: 5
Profundidade da solução: 85


In [ ]:
scenario_dls(30,4,3,10,50)

depth_limit = 10: nenhuma solução encontrada.
❌ Tempo de execução: 10.233753 s
